# 注意力机制（下）自注意力与 Transformer Encoder

chap8 上的加性注意力是 query 来自分类层 / decoder、key&value 来自 encoder——典型的 "encoder-decoder attention"。

**自注意力** (self-attention) 让序列里的每个位置都拿自己的 query 去 attend 所有位置的 key/value——序列内部任意两个位置直接通信，距离不再是障碍。这就是 Transformer 的核心。

本节自下而上：scaled dot-product attention → 多头注意力 → Transformer Encoder block → 在 chap8 上的情感任务上对比 BiLSTM + attention vs 纯 Transformer。

## 1. Scaled Dot-Product Attention

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$

$\sqrt{d_k}$ 缩放是为了避免维度高时点积变得过大、softmax 出现极端集中。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

def scaled_dot_attention(Q, K, V, mask=None):
    """Q,K,V: [..., L, d];  mask: [..., L_q, L_k] (True=valid)"""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)        # [..., L_q, L_k]
    if mask is not None:
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    attn = F.softmax(scores, dim=-1)
    return attn @ V, attn

# 验证：Q=K=V 时，每个位置应当注意到自己（如果其它位置很不像）
torch.manual_seed(0)
X = torch.eye(4) + 0.01 * torch.randn(4, 4)        # 近似 4 个正交单位向量
out, A = scaled_dot_attention(X, X, X)
print('attention matrix (diagonal should dominate):')
print(A.round(decimals=3))

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from nndl.runner import RunnerV3


## 2. 多头注意力

把 $Q, K, V$ 投影到 $h$ 个低维子空间，并行做 attention，再拼回来。让模型在不同子空间关注不同类型的关系（句法、语义、位置等）。

PyTorch 内置 `nn.MultiheadAttention`，但出于学习目的，我们手写一遍。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)

    def _split_heads(self, x):                # [B, L, embed] -> [B, n_heads, L, head_dim]
        B, L, _ = x.shape
        return x.reshape(B, L, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, key_padding_mask=None):
        # key_padding_mask: [B, L]  True=valid
        Q = self._split_heads(self.W_q(x))
        K = self._split_heads(self.W_k(x))
        V = self._split_heads(self.W_v(x))

        mask = None
        if key_padding_mask is not None:
            # 扩展到 [B, 1, 1, L]，broadcast 到所有 head 和 query 位置
            mask = key_padding_mask[:, None, None, :]

        out, attn = scaled_dot_attention(Q, K, V, mask=mask)   # out: [B, h, L, head_dim]
        B, h, L, d = out.shape
        out = out.transpose(1, 2).reshape(B, L, h * d)
        return self.W_o(out), attn                            # attn: [B, h, L, L]

mha = MultiHeadAttention(embed_dim=32, n_heads=4)
x = torch.randn(2, 7, 32)
out, attn = mha(x)
print('out:', tuple(out.shape), ' attn:', tuple(attn.shape))

## 3. 位置编码

自注意力本身**对位置无感**（permutation-equivariant）——打乱序列顺序，输出也只是相应地打乱。要让模型知道"第几个 token"，得额外加位置信号。

经典 sinusoidal positional encoding：
$$
PE_{pos, 2i} = \sin(pos / 10000^{2i/d}), \quad PE_{pos, 2i+1} = \cos(pos / 10000^{2i/d})
$$

In [ ]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)        # 不参与训练

    def forward(self, x):                     # x: [B, L, d_model]
        assert x.size(1) <= self.pe.size(0), (
            f"序列长度 {x.size(1)} 超过 max_len={self.pe.size(0)}"
        )
        return x + self.pe[:x.size(1)]

# 可视化前 100 个位置的前 16 维
pe = SinusoidalPE(d_model=64, max_len=100)
plt.figure(figsize=(10, 4))
plt.imshow(pe.pe[:100, :16].T, aspect='auto', cmap='RdBu_r')
plt.xlabel('position'); plt.ylabel('embedding dim'); plt.colorbar(); plt.title('sinusoidal positional encoding'); plt.show()

## 4. Transformer Encoder Block

标准结构：

```
x → MultiHeadAttn → +residual → LayerNorm
  → FeedForward    → +residual → LayerNorm
```

我们采用 **Pre-LN** 变体（LayerNorm 放在 sublayer 之前），训练更稳定，是 GPT-2 之后的现代默认。

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.GELU(),
            nn.Linear(ff_dim,  d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.attn(self.norm1(x), key_padding_mask)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

## 5. 完整模型：Transformer 分类器

embedding → +PE → N 个 Encoder block → 取 padding-aware 的平均 → 线性分类。

In [ ]:
VOCAB, PAD = 25, 0
POS = [1, 2, 3]; NEG = [4, 5, 6]; FILL = list(range(7, VOCAB))

class TransformerClassifier(nn.Module):
    def __init__(self, vocab=VOCAB, d_model=64, n_heads=4, ff_dim=128, n_layers=2,
                 n_class=2, dropout=0.1, max_len=64):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model, padding_idx=PAD)
        self.pe  = SinusoidalPE(d_model, max_len)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, ff_dim, dropout)
                                     for _ in range(n_layers)])
        self.norm  = nn.LayerNorm(d_model)
        self.fc    = nn.Linear(d_model, n_class)

    def forward(self, padded, lengths):
        mask = torch.arange(padded.size(1))[None, :] < lengths[:, None]   # [B, L] True=valid
        x = self.emb(padded)
        x = self.pe(x)
        for blk in self.blocks:
            x = blk(x, key_padding_mask=mask)
        x = self.norm(x)
        # padding-aware mean pooling
        m = mask.unsqueeze(-1).float()
        pooled = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)
        return self.fc(pooled)

m = TransformerClassifier()
print('params:', sum(p.numel() for p in m.parameters()))

## 6. 训练（情感任务，与 chap8 上同一个数据生成器）

In [ ]:
from nndl.data import (make_sentiment_sample as make_sample,
                       SentimentDS, sentiment_collate as collate,
                       VOCAB, PAD)

train_loader = DataLoader(SentimentDS(2000, seed=0), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader   = DataLoader(SentimentDS(500,  seed=1), batch_size=64, collate_fn=collate)



from nndl import accuracy

torch.manual_seed(0)
model = TransformerClassifier()
opt = optim.Adam(model.parameters(), lr=1e-3)
runner = RunnerV3(model, opt, nn.CrossEntropyLoss(),
                  metric_fn=accuracy, higher_is_better=True)
runner.fit(train_loader, dev_loader, num_epochs=5, log_every=1,
           grad_clip_norm=1.0)
accs = runner.history['dev_metric']

## 7. 看一下自注意力学到的关系

取一条 dev 样本，画第一层第一个 head 的注意力矩阵。如果模型学得好，**所有位置**的 query 都应该把高权重分配给 POS / NEG token 的位置。

In [ ]:
model.eval()
x_batch, len_batch, y_batch = next(iter(dev_loader))
x_one, len_one = x_batch[:1], len_batch[:1]

def token_kind(t):
    if t in POS: return '+'
    if t in NEG: return '-'
    return '.'

# 重新前向，并捕获第一个 block 的注意力
with torch.no_grad():
    x = model.emb(x_one); x = model.pe(x)
    mask = torch.arange(x_one.size(1))[None, :] < len_one[:, None]
    # 复刻 block forward, 拿 attention
    blk = model.blocks[0]
    h = blk.norm1(x)
    _, attn = blk.attn(h, key_padding_mask=mask)        # [B, n_heads, L, L]

L = len_one[0].item()
A = attn[0, 0, :L, :L]                                  # 第一个 head
tokens = x_one[0, :L].tolist()
kinds = [token_kind(t) for t in tokens]

plt.figure(figsize=(9, 8))
plt.imshow(A, cmap='hot', aspect='auto')
plt.colorbar(label='attention weight')
plt.xticks(range(L), [f'{k}{t}' for k, t in zip(kinds, tokens)], rotation=90, fontsize=7)
plt.yticks(range(L), [f'{k}{t}' for k, t in zip(kinds, tokens)], fontsize=7)
plt.xlabel('key (attended to)'); plt.ylabel('query (attending from)')
plt.title(f'block 0, head 0 attention  (true={y_batch[0].item()})')
plt.tight_layout(); plt.show()

亮列（key 维度）对应模型集中关注的位置；理想情况是 POS / NEG token 所在列普遍很亮。

## 8. 与 `nn.TransformerEncoder` 等价

PyTorch 自带 `nn.TransformerEncoderLayer` / `nn.TransformerEncoder`——实际项目直接用即可，比手写更优化（Flash Attention 等）。

In [ ]:
# 等价的 PyTorch 内置写法
layer = nn.TransformerEncoderLayer(
    d_model=64, nhead=4, dim_feedforward=128,
    dropout=0.1, batch_first=True, norm_first=True,        # Pre-LN
    activation='gelu',
)
encoder = nn.TransformerEncoder(layer, num_layers=2)

x = torch.randn(2, 10, 64)
pad_mask = torch.zeros(2, 10, dtype=torch.bool)
pad_mask[1, 7:] = True             # nn.Transformer: True 表示 *pad*（要忽略），与我们手写的相反
out = encoder(x, src_key_padding_mask=pad_mask)
print('nn.TransformerEncoder out:', tuple(out.shape))

**注意 mask 约定**：

- 我们手写：`mask[..., L] = True` 表示**有效**位置（attention 保留）
- `nn.TransformerEncoderLayer` 的 `src_key_padding_mask`：`True` 表示**应忽略**（padding 位置）

两种约定混用最容易翻车——切换 PyTorch 内置 API 时一定要记得 `~mask` 取反。

## 小结

- **Scaled dot-product attention**：$\text{softmax}(Q K^\top / \sqrt{d_k}) V$。$\sqrt{d_k}$ 缩放避免 softmax 集中。
- **多头**：把 query/key/value 切到 $h$ 个低维子空间并行做 attention，让模型在不同子空间关注不同类型的关系。
- **自注意力**：query=key=value=输入序列；序列内每两个位置直接通信，距离不再影响梯度。
- **位置编码**：自注意力本身位置无感，必须额外注入位置信号。Sinusoidal PE 不学习；learned PE 也常见；RoPE / ALiBi 是更现代的方案（chap8 PaddleEnding 之后）。
- **Transformer block**：MHA + FF + 两层 residual + LayerNorm。Pre-LN 比 Post-LN 训练更稳，是现代默认。
- **生产代码用 `nn.TransformerEncoderLayer`**——更快、更稳。但 mask 约定与教科书 / 大多数手写实现**相反**（True=ignore）。

---

# 案例：基于Transformer编码器的中文语义匹配（LCQMC）

前面用合成情感任务把自注意力、多头注意力和Transformer组块走了一遍。下面换一个更接近真实工程的任务——**文本语义匹配**：判断两句中文是不是表达同一个意思。

> 举一个直观的例子。下面三句话中，A和C在语义上更接近，但若仅按字面重合度判断，A看起来反而最像B：
>
> - A. 什么花一年四季都开？
> - B. 什么花生一年四季都开？
> - C. 哪些花可以全年开放？
>
> 语义匹配的目标恰好就是让模型从“意思上”判断两句话是否一致——A与B应输出0、A与C应输出1。

由于匹配本质上是一个二分类问题，只用Transformer的**编码器**部分即可。做法是把两个句子按`[CLS]+句子A+[SEP]+句子B+[SEP]`的模板拼成一句话：`[CLS]`是BERT风格的“句子特征位”，编码后它的向量代表整段语义；`[SEP]`是分隔符。每个字符同时携带三种编码——**词嵌入**（语义）、**位置编码**（顺序）、**分段编码**（0/1标记属于第一还是第二句），按位相加后送入编码器，最后取`[CLS]`向量过线性分类头得到“相似/不相似”。

> **数据说明**：本案例使用LCQMC（Large-scale Chinese Question Matching Corpus），来自百度知道领域的中文问题匹配数据集，规模为训练集238\,766条、验证集4\,401条、测试集4\,401条，对应的字表是BERT的中文字表（`vocab_size=21128`）。下面的数据加载单元会在若干候选位置自动寻找本地LCQMC数据（`practice-in-paddle/dataset/lcqmc`）与BERT字表（`bert-base-chinese/vocab.txt`）：找到就直接读取，并用`HAS_LCQMC`解锁后续训练/可视化；缺数据时只打印提示、不报错。为在CPU上也能跑通，训练默认只抽样3万条（`SAMPLE_TRAIN`），测试集保持全量做诚实评估。模型与算子单元（嵌入层、Transformer组块、`Model_Transformer`）是纯torch实现，本节末尾还会用一组合成`id`张量对它们做前向冒烟测试，无需LCQMC即可运行。


## A.1 数据集构建与本地数据加载

本实验采用**字符级**切分（每个汉字视为一个词）。先封装`LCQMCDataset`完成单条样本的预处理：把两句文本按字转成ID序列（未登录字用`[UNK]`代替），在开头加`[CLS]`、在两句之间和句末插入`[SEP]`，并额外生成一份`segment_ids`（0/1）标记每个词元属于第一句还是第二句。例如：

```
[CLS]什么花一年四季都开[SEP]什么花一年四季都是开的[SEP]
[ 0   0  0 0 0 0 0 0 0 0   0   1  1 1 1 1 1 1 1 1 1  1  1 ]
```

`collate_fn`负责成批：先按`max_seq_len`截断，再对短样本在尾部补`[PAD]`（ID为0）。

> 下面这个单元先定义`LCQMCDataset`/`collate_fn`（纯逻辑，会被后续合成测试复用），再在若干候选位置查找本地LCQMC（gzip）与BERT字表：备齐则读出`train_data`/`dev_data`/`test_data`/`word2id_dict`并置`HAS_LCQMC=True`；缺数据时只打印提示、不报错。BERT中文字表的`[CLS]`/`[SEP]`等特殊符号已按行号收录，逐字查表即可。


In [ ]:
# === LCQMC 本地数据加载（gzip）+ 词表（BERT 优先 / 字符级兜底）===
# 数据加载、DataLoader、训练单元都在本地 LCQMC 备齐后可运行；
# LCQMCDataset / collate_fn 的定义是纯逻辑，也被本节末尾的合成冒烟测试复用。
import os
import gzip
import random
import torch
from torch.utils.data import Dataset, DataLoader


class LCQMCDataset(Dataset):
    def __init__(self, data, word2id_dict):
        self.word2id_dict = word2id_dict
        self.examples = data
        self.cls_id = self.word2id_dict['[CLS]']   # [CLS] 的 id
        self.sep_id = self.word2id_dict['[SEP]']   # [SEP] 的 id

    def __getitem__(self, idx):
        example = self.examples[idx]
        text, segment, label = self.words_to_id(example)
        return text, segment, label

    def __len__(self):
        return len(self.examples)

    def words_to_id(self, example):
        text_a, text_b, label = example
        # 逐字转 id，未登录字用 [UNK]
        input_ids_a = [self.word2id_dict[item] if item in self.word2id_dict
                       else self.word2id_dict['[UNK]'] for item in text_a]
        input_ids_b = [self.word2id_dict[item] if item in self.word2id_dict
                       else self.word2id_dict['[UNK]'] for item in text_b]
        # 拼接 [CLS] A [SEP] B [SEP]
        input_ids = [self.cls_id] + input_ids_a + [self.sep_id] + input_ids_b + [self.sep_id]
        # 分段编码：第一句（含 [CLS] 与第一个 [SEP]）记 0，第二句记 1
        segment_ids = [0] * (len(input_ids_a) + 2) + [1] * (len(input_ids_b) + 1)
        return input_ids, segment_ids, int(label)

    @property
    def label_list(self):
        return ['0', '1']   # 0 不相似，1 相似


def collate_fn(batch_data, pad_val=0, max_seq_len=512):
    input_ids, segment_ids, labels = [], [], []
    max_len = 0
    for example in batch_data:
        input_id, segment_id, label = example
        input_ids.append(input_id[:max_seq_len])      # 截断
        segment_ids.append(segment_id[:max_seq_len])
        labels.append(label)
        max_len = max(max_len, len(input_id))
    for i in range(len(labels)):                       # 尾部补 [PAD]
        input_ids[i] = input_ids[i] + [pad_val] * (max_len - len(input_ids[i]))
        segment_ids[i] = segment_ids[i] + [pad_val] * (max_len - len(segment_ids[i]))
    return torch.tensor(input_ids), torch.tensor(segment_ids), torch.tensor(labels)


# ---- 本地数据 / 词表：在若干候选位置里找 LCQMC 目录和 BERT 中文字表 ----
# LCQMC gzip：每个 split 一个 <split>.txt.gz，每行 `句子1\t句子2\t标签`（标签 0/1）。
_LCQMC_CANDIDATES = [
    os.path.join("..", "datasets", "lcqmc"),
    os.path.join("..", "..", "..", "practice-in-paddle", "dataset", "lcqmc"),
    os.path.join("dataset", "lcqmc"),
]
DATA_DIR = next((p for p in _LCQMC_CANDIDATES
                 if os.path.isfile(os.path.join(p, "train.txt.gz"))), _LCQMC_CANDIDATES[0])
HAS_LCQMC = os.path.isfile(os.path.join(DATA_DIR, "train.txt.gz"))

# 可选 BERT 中文 vocab.txt（vocab_size=21128）：放在任一候选路径即“精确复刻”，否则字符级自建兜底。
_VOCAB_CANDIDATES = [
    os.path.join("..", "datasets", "bert-base-chinese", "vocab.txt"),
    os.path.join("..", "..", "..", "practice-in-paddle", "dataset", "bert-base-chinese", "vocab.txt"),
    os.path.join(DATA_DIR, "vocab.txt"),
]

# 抽样提速：全量 train=238766 在 CPU 上很慢，默认只抽 3 万训练 + 3 千验证即可画出
# 收敛曲线与注意力热图；测试集保持全量做诚实评估。
SAMPLE_TRAIN = 30000
SAMPLE_DEV = 3000
SAMPLE_TEST = None     # None 表示全量测试集
SAMPLE_SEED = 2021


def load_lcqmc_split(name):
    """读取一个 split，返回 [(text_a, text_b, label), ...]。"""
    rows = []
    with gzip.open(os.path.join(DATA_DIR, name), "rt", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 3:
                continue
            text_a, text_b, label = parts
            rows.append((text_a, text_b, int(label)))
    return rows


def build_char_vocab(train_data):
    """从训练集字符级自建词表（特殊符号在前，与 toy_vocab 顺序一致）。

    等价于 LCQMCDataset.words_to_id 对 word2id_dict 的用法（逐字符查表），
    只是把缺失的 BERT vocab.txt 换成训练集自建表。
    """
    word2id_dict = {"[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3}
    for text_a, text_b, _ in train_data:
        for ch in text_a:
            if ch not in word2id_dict:
                word2id_dict[ch] = len(word2id_dict)
        for ch in text_b:
            if ch not in word2id_dict:
                word2id_dict[ch] = len(word2id_dict)
    return word2id_dict


def load_bert_vocab(path):
    """读取 BERT 风格 vocab.txt：每行一个 token，行号即 id（依名取特殊符号）。"""
    word2id_dict = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            word2id_dict[line.rstrip("\n")] = i
    return word2id_dict


def build_vocab(train_data):
    """优先 BERT 中文词表（精确复刻 vocab_size=21128）；找不到则字符级自建（fallback）。"""
    for p in _VOCAB_CANDIDATES:
        if os.path.isfile(p):
            w2i = load_bert_vocab(p)
            if all(t in w2i for t in ("[PAD]", "[UNK]", "[CLS]", "[SEP]")):
                print(f"词表：BERT 中文字表 {p}（size={len(w2i)}）")
                return w2i
    w2i = build_char_vocab(train_data)
    print(f"词表：字符级自建（size={len(w2i)}）")
    return w2i


def subsample(data, n, seed):
    """随机抽样 n 条（n=None 或超过总量则用全量），固定种子保证可复现。"""
    if n is None or n >= len(data):
        return data
    rng = random.Random(seed)
    return rng.sample(data, n)


if HAS_LCQMC:
    train_data = subsample(load_lcqmc_split("train.txt.gz"), SAMPLE_TRAIN, SAMPLE_SEED)
    dev_data = subsample(load_lcqmc_split("dev.txt.gz"), SAMPLE_DEV, SAMPLE_SEED)
    test_data = subsample(load_lcqmc_split("test.txt.gz"), SAMPLE_TEST, SAMPLE_SEED)
    word2id_dict = build_vocab(train_data)
    print("数据目录：", DATA_DIR)
    print(f"训练集样本数：{len(train_data)}  验证集：{len(dev_data)}  测试集：{len(test_data)}")
    print("词表大小：", len(word2id_dict),
          "（21128 表示用上了 BERT 中文字表，否则为训练集字符级自建）")
    print("数据示例：", train_data[0])
else:
    print("[需 LCQMC 数据] 未在以下位置找到 train.txt.gz：", _LCQMC_CANDIDATES,
          "—— 准备好 practice-in-paddle/dataset/lcqmc 后即可解锁后续训练/可视化 cell。")


## A.2 嵌入层：词嵌入 + 分段编码 + 位置编码

嵌入层把token序列变成向量序列。除了常规词向量，还需要两类额外编码：

1. **位置编码**：注意力对位置无感，移除CNN/RNN后必须显式注入顺序信息；
2. **分段编码**：输入是两段文本拼接，给每个词元一个0/1标签区分句子归属。

三种编码同维度，按位相加后再过一次LayerNorm和Dropout，得到最终输入向量。下面四个算子都是纯torch实现，可独立运行。

> **一处实现修正**：原书把`PositionalEmbedding`写成一个随机初始化的`nn.Embedding`，> 同时单独给出`get_sinusoid_encoding`，但并未把正余弦编码真正写进嵌入表（表里始终是随机值）。> 这里把`get_sinusoid_encoding`的结果`copy_`进权重并冻结梯度，才真正实现了“确定性正余弦位置编码”。


In [ ]:
import numpy as np
import torch.nn as nn
import torch.nn.functional as F


class WordEmbedding(nn.Module):
    """输入编码：按原始 Transformer 约定，前向时乘上 sqrt(emb_size)。"""
    def __init__(self, vocab_size, emb_size, padding_idx=0):
        super().__init__()
        self.emb_size = emb_size
        self.word_embedding = nn.Embedding(vocab_size, emb_size, padding_idx=padding_idx)

    def forward(self, word):
        return self.emb_size ** 0.5 * self.word_embedding(word)


class SegmentEmbedding(nn.Module):
    """分段编码：本质就是“词表只有 2”的查表（0=第一句，1=第二句）。"""
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.emb_size = emb_size
        self.seg_embedding = nn.Embedding(vocab_size, emb_size)

    def forward(self, word):
        return self.seg_embedding(word)


def get_sinusoid_encoding(seq_len, hidden_size):
    """正余弦位置编码表：偶数维 sin、奇数维 cos，频率随维度指数衰减。"""
    def cal_angle(pos, hidden_idx):
        return pos / np.power(10000, 2 * (hidden_idx // 2) / hidden_size)

    def get_pos_angle_vec(pos):
        return [cal_angle(pos, hidden_j) for hidden_j in range(hidden_size)]

    sinusoid = np.array([get_pos_angle_vec(pos_t) for pos_t in range(seq_len)])
    sinusoid[:, 0::2] = np.sin(sinusoid[:, 0::2])   # 偶数维正弦
    sinusoid[:, 1::2] = np.cos(sinusoid[:, 1::2])   # 奇数维余弦
    return sinusoid.astype("float32")


class PositionalEncoding(nn.Module):
    """位置编码算子：用正余弦表初始化并冻结梯度（确定性变换，不参与训练）。"""
    def __init__(self, max_length, emb_size):
        super().__init__()
        self.emb_size = emb_size
        self.pos_encoder = nn.Embedding(max_length, self.emb_size)
        # 把正余弦编码真正写进嵌入表（原书漏了这一步），再冻结
        pe = torch.from_numpy(get_sinusoid_encoding(max_length, emb_size))
        self.pos_encoder.weight.data.copy_(pe)
        self.pos_encoder.weight.requires_grad = False

    def forward(self, pos):
        return self.pos_encoder(pos)


# 快速验证：词嵌入对 [PAD]=0 给出全 0 向量
torch.manual_seed(2021)
word_embed = WordEmbedding(10, 4)
print("WordEmbedding([1,0,2]) =\n", word_embed(torch.tensor([1, 0, 2])).detach().numpy())
# 位置编码取值落在 [-1, 1]
pe_table = get_sinusoid_encoding(100, 64)
print("位置编码表 shape:", pe_table.shape, " 取值范围:", round(float(pe_table.min()), 3),
      "~", round(float(pe_table.max()), 3))


In [ ]:
class TransformerEmbeddings(nn.Module):
    """嵌入层汇总：输入编码 + 分段编码 + 位置编码，相加后 LayerNorm + Dropout。"""
    def __init__(self, vocab_size, hidden_size=768, hidden_dropout_prob=0.1,
                 seq_len=512, segment_size=2):
        super().__init__()
        self.word_embeddings = WordEmbedding(vocab_size, hidden_size)
        self.position_encoding = PositionalEncoding(seq_len, hidden_size)
        self.segment_embeddings = SegmentEmbedding(segment_size, hidden_size)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(hidden_dropout_prob)

    def forward(self, input_ids, segment_ids=None, position_ids=None):
        if position_ids is None:
            # 由 [1,1,...] 累加得到 [0,1,2,...] 的位置索引
            ones = torch.ones_like(input_ids, dtype=torch.int64)
            seq_length = torch.cumsum(ones, dim=-1)
            position_ids = seq_length - ones
            position_ids.requires_grad = False
        input_embedings = self.word_embeddings(input_ids)
        segment_embeddings = self.segment_embeddings(segment_ids)
        position_encoding = self.position_encoding(position_ids)
        # 三种编码按位相加
        embeddings = input_embedings + segment_embeddings + position_encoding
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings


## A.3 Transformer 组块与语义匹配模型

Transformer编码器是若干**组块**的堆叠，每块由“多头自注意力、加与规范（Add\&Norm）、逐位前馈、加与规范”四部分组成，两条残差路径让深层网络训得稳。这里多头自注意力直接复用PyTorch内置的`nn.MultiheadAttention`（注意是小写`h`，`batch_first=True`），`key_padding_mask`里`True`表示该位置是`[PAD]`、会被掩蔽。

模型`Model_Transformer`把嵌入层、若干Transformer组块和分类头串起来：取`[CLS]`（第0个位置）的输出向量，过一层`Dense + Tanh`再接`classifier`得到二分类得分。这些都是纯torch实现，可在合成数据上直接前向。


In [ ]:
class PositionwiseFFN(nn.Module):
    """逐位前馈层：两层全连接 + ReLU，D' 通常取 4D。"""
    def __init__(self, input_size, mid_size, dropout=0.1):
        super().__init__()
        self.W_1 = nn.Linear(input_size, mid_size)
        self.W_2 = nn.Linear(mid_size, input_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        return self.W_2(self.dropout(F.relu(self.W_1(X))))


class AddNorm(nn.Module):
    """加与规范化（Post-LN 风格）：H <- LayerNorm(X + Dropout(H))。"""
    def __init__(self, size, dropout_rate):
        super().__init__()
        self.layer_norm = nn.LayerNorm(size)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, X, H):
        H = X + self.dropout(H)
        return self.layer_norm(H)


class TransformerBlock(nn.Module):
    """Transformer 组块：多头自注意力 + Add&Norm + 逐位前馈 + Add&Norm。"""
    def __init__(self, input_size, head_num, ffn_size, dropout=0.1,
                 attn_dropout=None, act_dropout=None):
        super().__init__()
        self.input_size = input_size
        self.head_num = head_num
        self.ffn_size = ffn_size
        self.dropout = dropout
        self.attn_dropout = dropout if attn_dropout is None else attn_dropout
        self.act_dropout = dropout if act_dropout is None else act_dropout
        # 多头自注意力层，使用 PyTorch 内置 nn.MultiheadAttention（h 小写）
        self.multi_head_attention = nn.MultiheadAttention(
            self.input_size, self.head_num, dropout=self.attn_dropout, batch_first=True)
        self.ffn = PositionwiseFFN(self.input_size, self.ffn_size, self.act_dropout)
        self.addnorm = AddNorm(self.input_size, self.dropout)

    def forward(self, X, key_padding_mask=None):
        # 自注意力的 query/key/value 都是 X；key_padding_mask 里 True 表示 [PAD]
        # average_attn_weights=False 让返回的是“逐头”权重 [B, n_heads, L, L]，
        # 以便后面画出书中 head-1..head-4 共 4 张热图；该参数不进入前向输出、不改变训练结果。
        X_atten, atten_weights = self.multi_head_attention(
            X, X, X, key_padding_mask=key_padding_mask,
            need_weights=True, average_attn_weights=False)
        X = self.addnorm(X, X_atten)      # 第一次 Add&Norm
        X_ffn = self.ffn(X)
        X = self.addnorm(X, X_ffn)        # 第二次 Add&Norm
        return X, atten_weights


In [ ]:
class Model_Transformer(nn.Module):
    """基于 Transformer 编码器的语义匹配模型：嵌入层 + N 个组块 + [CLS] 二分类头。"""
    def __init__(self, vocab_size, n_block=2, hidden_size=768, heads_num=12,
                 intermediate_size=3072, hidden_dropout=0.1, attention_dropout=0.1,
                 act_dropout=0, seq_len=512, num_classes=2):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_block = n_block
        self.hidden_size = hidden_size
        self.heads_num = heads_num
        self.intermediate_size = intermediate_size
        self.hidden_dropout = hidden_dropout
        self.attention_dropout = attention_dropout
        self.act_dropout = act_dropout
        self.seq_len = seq_len
        self.num_classes = num_classes
        # 输入编码、分段编码、位置编码
        self.embeddings = TransformerEmbeddings(
            self.vocab_size, self.hidden_size, self.hidden_dropout, self.seq_len)
        # 堆叠 n_block 个 Transformer 组块
        self.layers = nn.ModuleList([])
        for i in range(n_block):
            encoder_layer = TransformerBlock(
                hidden_size, heads_num, intermediate_size,
                dropout=hidden_dropout, attn_dropout=attention_dropout, act_dropout=act_dropout)
            self.layers.append(encoder_layer)
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.activation = nn.Tanh()
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, segment_ids, position_ids=None, attention_mask=None):
        # 构造 key_padding_mask：[batch, length] 布尔张量，True 表示 [PAD]
        if attention_mask is None:
            attention_mask = (input_ids == 0)
        embedding_output = self.embeddings(
            input_ids=input_ids, position_ids=position_ids, segment_ids=segment_ids)
        sequence_output = embedding_output
        self._attention_weights = []
        for i, encoder_layer in enumerate(self.layers):
            sequence_output, atten_weights = encoder_layer(
                sequence_output, key_padding_mask=attention_mask)
            self._attention_weights.append(atten_weights)
        # 取第 0 个位置（[CLS]）的向量作为句向量
        first_token_tensor = sequence_output[:, 0]
        pooled_output = self.dense(first_token_tensor)
        pooled_output = self.activation(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

    @property
    def attention_weights(self):
        # 每个组块一份注意力权重，组成列表
        return self._attention_weights


## A.4 合成冒烟测试（无需 LCQMC，可直接运行）

下面用一个手工小词表造两条假样本，把`LCQMCDataset`→`collate_fn`→`Model_Transformer`整条链路跑一遍前向+反向，确认算子拼装正确。真实实验只需把这里的小词表/小配置换成LCQMC的字表（`vocab_size=21128`）和书中超参（`n_block=1`、`heads_num=4`、`hidden_size=768`）即可。


In [ ]:
# === 可运行：用合成 id 张量验证整条链路 ===
torch.manual_seed(2021)
# 手工小词表（真实实验请换成 LCQMC 的 BERT 字表）
toy_vocab = {'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3}
for ch in "什么花一年四季都开是的哪些可以全放":
    if ch not in toy_vocab:
        toy_vocab[ch] = len(toy_vocab)
toy_data = [("什么花一年四季都开", "什么花一年四季都是开的", 1),
            ("什么花一年四季都开", "哪些花可以全年开放", 1)]

toy_ds = LCQMCDataset(toy_data, toy_vocab)
print("样本0（input_ids, segment_ids, label）:")
print("  ", toy_ds[0])
toy_loader = DataLoader(toy_ds, batch_size=2, collate_fn=collate_fn, shuffle=False)
input_ids, segment_ids, labels = next(iter(toy_loader))
print("collate 后 input_ids:", tuple(input_ids.shape),
      " segment_ids:", tuple(segment_ids.shape), " labels:", labels.tolist())

# 迷你配置，仅验证前向/反向是否打通
model = Model_Transformer(vocab_size=len(toy_vocab), n_block=1, hidden_size=32,
                          heads_num=4, intermediate_size=64, seq_len=64, num_classes=2)
logits = model(input_ids, segment_ids)
print("logits:", tuple(logits.shape),
      " 注意力权重块数:", len(model.attention_weights),
      " block0:", tuple(model.attention_weights[0].shape))
loss = F.cross_entropy(logits, labels)
loss.backward()
print("forward + backward OK, loss =", round(loss.item(), 4),
      " 参数量:", sum(p.numel() for p in model.parameters()))


## A.5 训练、评价与注意力可视化（本地 LCQMC）

下面给出端到端的训练、评价和注意力可视化代码：它们由A.1构建的`train_data`等和真实字表`word2id_dict`驱动，用`HAS_LCQMC`保护——备齐LCQMC即可整段跑通，缺数据时跳过。CPU上约7\,min/回合，可调小`SAMPLE_TRAIN`/`num_epochs`提速。

训练沿用工程化的`RunnerV3`：损失取交叉熵、优化器用AdamW（学习率$5\times10^{-5}$、对bias与LayerNorm权重关闭权重衰减），训练3个回合并保留验证集最优快照。书中报告的测试集准确率约为0.665——从零训练、单组块、问句又短，容易欠拟合；本节着重展示流程，更高精度通常要用预训练BERT来初始化。

> 训练好的验证集最优快照存到`./checkpoint/model_best.pt`，评价时再加载它在测试集上打分。因为A.3的`TransformerBlock`用了`average_attn_weights=False`，`attention_weights[0][0]`是逐头的`[n_heads, L, L]`，可分别画出4个注意力头的热图。


In [ ]:
# === 真实训练 / 评价（本地 LCQMC，CPU 约 7 min/epoch，可调小 SAMPLE_TRAIN / EPOCHS）===
# 训练配置照搬书 1730-1761：n_block=1、heads_num=4、hidden_size=768、AdamW(lr=5e-5,
# 对 bias 与 LayerNorm 关权重衰减)、3 个回合、grad_clip_norm=1.0、保留验证集最优快照。
# 书中报告测试集准确率约 0.665——从零训练、单组块、问句又短，容易欠拟合；本节重在流程，
# 更高精度通常要用预训练 BERT 初始化。
import os
import sys
import torch
import torch.nn as nn

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from nndl.runner import RunnerV3
from nndl import accuracy

if HAS_LCQMC:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        torch.set_num_threads(os.cpu_count() or 4)
    torch.manual_seed(2021)
    print(f"device={device}  threads={torch.get_num_threads()}")

    label_list = ['0', '1']
    num_classes = len(label_list)
    heads_num = 4
    batch_size = 32
    max_seq_len = 512
    num_epochs = 3

    def coll(b):
        return collate_fn(b, pad_val=word2id_dict["[PAD]"], max_seq_len=max_seq_len)

    train_dataset = LCQMCDataset(train_data, word2id_dict)
    dev_dataset = LCQMCDataset(dev_data, word2id_dict)
    test_dataset = LCQMCDataset(test_data, word2id_dict)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=coll, shuffle=True)
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, collate_fn=coll, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=coll, shuffle=False)

    # 模型（默认 hidden_size=768, intermediate_size=3072, n_block=1）
    model = Model_Transformer(vocab_size=len(word2id_dict), n_block=1,
                              num_classes=num_classes, heads_num=heads_num,
                              seq_len=max_seq_len)

    # 优化器：AdamW，bias/LayerNorm 关闭权重衰减
    criterion = nn.CrossEntropyLoss()
    metric = accuracy
    decay_params, no_decay_params = [], []
    for n, p in model.named_parameters():
        if any(nd in n for nd in ["bias", "norm"]):
            no_decay_params.append(p)
        else:
            decay_params.append(p)
    optimizer = torch.optim.AdamW([
        {"params": decay_params, "weight_decay": 0.01},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=5e-5)

    # 训练（RunnerV3 接管 batch 搬运到 device）
    os.makedirs("./checkpoint", exist_ok=True)
    ckpt = "./checkpoint/model_best.pt"
    runner = RunnerV3(model, optimizer, criterion, metric_fn=metric, device=device)
    print("开始训练 ...")
    runner.fit(train_loader, dev_loader, num_epochs=num_epochs, log_every=1,
               best_path=ckpt, grad_clip_norm=1.0)

    # 评价：加载验证集最优快照，在测试集上评估
    runner.load(ckpt)
    test_loss, test_acc = runner.evaluate(test_loader)
    print(f"Evaluate on test set, Accuracy: {test_acc:.5f}")   # 书中约 0.66485
else:
    print("[需 LCQMC 数据] 跳过训练与评价。备齐 practice-in-paddle/dataset/lcqmc 后重跑本 cell。")


画出训练损失与验证准确率随回合的变化（书图 `att-loss-acc3`）。

In [ ]:
# === 训练损失 / 准确率曲线（对应书图 att-loss-acc3）===
# 取 RunnerV3.history 的逐回合记录：左轴 train_loss / dev_loss，右轴 dev_metric（准确率）。
if HAS_LCQMC:
    import matplotlib.pyplot as plt

    history = runner.history
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax1 = plt.subplots(figsize=(7, 5))
    ax1.plot(epochs, history["train_loss"], "o-", color="tab:blue", label="train loss")
    if history.get("dev_loss"):
        ax1.plot(epochs, history["dev_loss"], "s--", color="tab:cyan", label="dev loss")
    ax1.set_xlabel("epoch")
    ax1.set_ylabel("loss")
    ax1.set_xticks(list(epochs))

    ax2 = ax1.twinx()
    if history.get("dev_metric"):
        ax2.plot(epochs, history["dev_metric"], "^-", color="tab:red", label="dev accuracy")
        ax2.set_ylabel("accuracy")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")
    ax1.set_title("Transformer on LCQMC — loss & accuracy")
    fig.tight_layout()
    plt.show()
else:
    print("[需 LCQMC 数据] 跳过损失/准确率曲线。")


取一对样本（“电脑怎么录像？” / “如何在计算机上录视频”），画出 block-0 这 4 个注意力头的权重热图（书图 `att-vis3-0..3`）。因为前面把 `average_attn_weights=False`，`attention_weights[0][0]` 是逐头的 `[n_heads, L, L]`，对应 head-1..head-4 各一张。亮列（key 维度）就是该头集中关注的位置。

In [ ]:
# === 多头自注意力热图（对应书图 att-vis3-0..3，同一样本、block-0 的 4 个头）===
# 样本/取权重方式与书 1832-1864 一致：texts=['CLS']+句A+['SEP']+句B+['SEP']，
# atten=model.attention_weights[0][0]；因前面 average_attn_weights=False，atten 形状为
# [n_heads, L, L]，逐头各画一张。
VIS_TEXT_A = "电脑怎么录像？"
VIS_TEXT_B = "如何在计算机上录视频"

if HAS_LCQMC:
    import matplotlib
    import matplotlib.pyplot as plt

    cls_id = word2id_dict["[CLS]"]
    sep_id = word2id_dict["[SEP]"]
    text_a, text_b = VIS_TEXT_A, VIS_TEXT_B
    texts = ['CLS'] + list(text_a) + ['SEP'] + list(text_b) + ['SEP']
    input_ids_a = [word2id_dict.get(c, word2id_dict["[UNK]"]) for c in text_a]
    input_ids_b = [word2id_dict.get(c, word2id_dict["[UNK]"]) for c in text_b]
    input_ids = [cls_id] + input_ids_a + [sep_id] + input_ids_b + [sep_id]
    segment_ids = [0] * (len(input_ids_a) + 2) + [1] * (len(input_ids_b) + 1)
    input_ids = torch.tensor([input_ids], device=device)
    segment_ids = torch.tensor([segment_ids], device=device)

    model.eval()
    with torch.no_grad():
        _ = model(input_ids, segment_ids)
    # 第 0 块、批内第 0 条 -> [n_heads, L, L]
    atten = model.attention_weights[0][0].cpu().numpy()
    n_heads = atten.shape[0]

    # 中文字体：尽量挑一个能渲染中文的，挑不到就退回默认（不影响出图）
    for fam in ["Microsoft YaHei", "SimHei", "SimSun", "DejaVu Sans"]:
        try:
            matplotlib.rcParams["font.sans-serif"] = [fam]
            matplotlib.rcParams["axes.unicode_minus"] = False
            break
        except Exception:
            continue

    L = len(texts)
    for h in range(n_heads):
        head = atten[h][:L, :L]
        plt.figure(figsize=(7, 6))
        plt.imshow(head, cmap="hot", aspect="auto")
        plt.colorbar(label="attention weight")
        plt.xticks(range(L), texts, rotation=90, fontsize=7)
        plt.yticks(range(L), texts, fontsize=7)
        plt.xlabel("key (attended to)")
        plt.ylabel("query (attending from)")
        plt.title(f"LCQMC Transformer block-0  head-{h + 1}")
        plt.tight_layout()
        plt.show()
else:
    print("[需 LCQMC 数据] 跳过注意力可视化。")


---

# 现代 Transformer 的若干优化

前面实现的是Vaswani等人2017年提出的“原汁原味”Transformer编码器：正余弦位置编码、Post-LayerNorm、纯dense FFN。但从那时到现在，架构经历了多轮工程化迭代——主流大模型（Llama、Qwen、GPT等）用的早已是**现代版本**。本节挑出影响最深远的几个改造，都用合成小张量做可运行演示。


## B.1 Pre-LayerNorm vs Post-LayerNorm

设子层为$\mathbf{H}=f(\mathbf{X})$，两种规范化位置：

$$\text{Post-LN}:\quad \mathbf{x}\leftarrow\text{LayerNorm}\big(\mathbf{x}+\text{Sublayer}(\mathbf{x})\big),$$
$$\text{Pre-LN}:\quad \mathbf{x}\leftarrow\mathbf{x}+\text{Sublayer}\big(\text{LayerNorm}(\mathbf{x})\big).$$

看似只是换了一处规范化的位置，但层数很深时影响巨大：Post-LN容易梯度爆炸/消失、必须配合细致预热；Pre-LN则形成清晰的“残差流”（residual stream），训练稳定、对学习率不敏感、不需要繁琐预热。本notebook前面的`TransformerBlock`用的正是Pre-LN；上面LCQMC案例里的`AddNorm`则是Post-LN。


In [ ]:
class PostLNSublayer(nn.Module):
    """Post-LN：x <- LayerNorm(x + Sublayer(x))。"""
    def __init__(self, d_model):
        super().__init__()
        self.sublayer = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(x + self.sublayer(x))


class PreLNSublayer(nn.Module):
    """Pre-LN：x <- x + Sublayer(LayerNorm(x))。"""
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.sublayer = nn.Linear(d_model, d_model)

    def forward(self, x):
        return x + self.sublayer(self.norm(x))


torch.manual_seed(2021)
d_model = 16
x = torch.randn(2, 5, d_model)
print("Post-LN out:", tuple(PostLNSublayer(d_model)(x).shape))
print("Pre-LN  out:", tuple(PreLNSublayer(d_model)(x).shape))


## B.2 旋转位置编码 RoPE

**旋转位置编码**（Rotary Position Embedding，RoPE）由Su等人2021年提出，是Llama、GPT-NeoX等模型选用的方案。与正余弦位置编码“加到输入”不同，RoPE直接**对查询和键向量旋转**来注入位置信息。把$d$维向量按$d/2$对$(x_{2i},x_{2i+1})$分组，在2D平面上各自旋转角度$m\theta_i$（$m$为位置索引，$\theta_i=10000^{-2i/d}$）：

$$\begin{pmatrix} x'_{2i}\\ x'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i)\\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix}\begin{pmatrix} x_{2i}\\ x_{2i+1} \end{pmatrix}.$$

它的好处：**相对位置自然成立**（打分只取决于相对距离）、**长上下文外推友好**（可借NTK插值、YaRN外推）、**零额外参数**（纯几何变换）。工程上常用复数实现：把相邻两维合并成复数，乘上$e^{im\theta_i}$即等价于旋转。


In [ ]:
def precompute_rope_freqs(dim, max_seq_len, base=10000.0):
    """预计算 RoPE 的复数频率 e^{i theta}。"""
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len).float()                   # [max_seq_len]
    freqs = torch.outer(t, inv_freq)                        # [max_seq_len, dim/2]
    return torch.polar(torch.ones_like(freqs), freqs)       # complex64


def apply_rope(x, freqs_complex):
    """对 q 或 k 应用 RoPE。x: [..., seq, dim]。"""
    # 把 x 相邻两维合并成复数：[..., seq, dim/2]
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    seq_len = x.shape[-2]
    f = freqs_complex[:seq_len]                             # [seq, dim/2]
    while f.dim() < x_complex.dim():                        # 广播到 batch / head 维
        f = f.unsqueeze(0)
    x_rotated = torch.view_as_real(x_complex * f).flatten(-2)
    return x_rotated.type_as(x)


torch.manual_seed(2021)
B, H, L, D = 2, 4, 8, 16
q = torch.randn(B, H, L, D)
k = torch.randn(B, H, L, D)
freqs = precompute_rope_freqs(D, max_seq_len=32)
q_rot, k_rot = apply_rope(q, freqs), apply_rope(k, freqs)
print("freqs:", tuple(freqs.shape), freqs.dtype)
print("q_rot:", tuple(q_rot.shape), " k_rot:", tuple(k_rot.shape))
# 旋转不改变向量长度（保范数）
print("norm 保持不变:", torch.allclose(q.norm(dim=-1), q_rot.norm(dim=-1), atol=1e-4))


## B.3 三种架构范式

Transformer家族当前主要有三种典型架构：

| 架构 | 典型模型 | 注意力 | 适用任务 |
|---|---|---|---|
| Encoder-only | BERT、RoBERTa | 双向（全局）注意力 | 理解任务：分类、抽取、检索 |
| Decoder-only | GPT、Llama、Qwen | 因果（单向）注意力 | 生成任务：续写、对话、推理 |
| Encoder-Decoder | 原始Transformer、T5、BART | encoder双向 + decoder因果 + cross-attention | 序列到序列：翻译、摘要 |

**为什么近年decoder-only成为主流？** 从理论容量看encoder-decoder更强，但decoder-only有几个压倒性的工程优势：结构最简（所有token共享一套参数）、训练目标统一（next-token prediction可直接吃任意原始文本，不需平行语料）、使用方式统一（配合in-context learning，few-shot prompt即可覆盖大部分NLP任务）、推理高效（自回归天然适合KV cache）。本节上面的LCQMC用的是encoder-only；下一章从零搭nanoGPT用的是decoder-only。


## B.4 混合专家 MoE

随着模型规模膨胀，dense Transformer的计算量按$O(N\cdot d_{\text{model}}^2)$迅速增长。**混合专家**（Mixture of Experts，MoE）把单一大FFN换成一组小FFN+一个路由器——每个词元动态挑其中1$\sim$2个专家参与计算，激活参数远小于总参数：

$$\text{MoE}(\mathbf{x})=\sum_{e=1}^{E} g_e(\mathbf{x})\cdot E_e(\mathbf{x}),$$

其中$g_e(\mathbf{x})$是路由器输出的稀疏门控权重（通常只在top-$k$个专家上非零），$E_e$是第$e$个专家FFN。代表模型有Mixtral 8x7B（8专家、每token选2个）、DeepSeek-V3等。下面是一个Top-$k$路由的最小可运行实现，仅供理解原理（真实MoE还要处理负载均衡auxiliary loss、专家并行的all-to-all通信、专家容量上限等工程难点）。


In [ ]:
class MoEFeedForward(nn.Module):
    """简化版 Mixture-of-Experts FFN，Top-2 路由（仅供理解原理）。"""

    def __init__(self, n_embd, n_experts=4, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        # n_experts 个独立 FFN
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
                          nn.Linear(4 * n_embd, n_embd))
            for _ in range(n_experts)
        ])
        # 路由器：n_embd -> n_experts
        self.gate = nn.Linear(n_embd, n_experts, bias=False)

    def forward(self, x):
        # x: [B, T, C]
        B, T, C = x.shape
        # 1. 每个 token 选 top-k 个专家
        logits = self.gate(x)                                   # [B, T, n_experts]
        weights, idx = torch.topk(logits, self.top_k, dim=-1)   # [B, T, top_k]
        weights = F.softmax(weights, dim=-1)                    # 归一化
        # 2. 把每个 token 路由到选中的专家
        out = torch.zeros_like(x)
        for e in range(self.n_experts):
            mask = (idx == e).any(dim=-1)                       # 哪些 token 选了第 e 个专家
            if mask.any():
                w = (weights * (idx == e).float()).sum(dim=-1)  # 对应位置的路由权重
                out[mask] = out[mask] + (w[mask].unsqueeze(-1) * self.experts[e](x[mask]))
        return out


torch.manual_seed(2021)
moe = MoEFeedForward(n_embd=16, n_experts=4, top_k=2)
x = torch.randn(2, 5, 16)
y = moe(x)
print("MoE in:", tuple(x.shape), " out:", tuple(y.shape))
expert_params = sum(p.numel() for p in moe.experts[0].parameters())
print(f"每专家参数 {expert_params}，4 专家共 {expert_params * 4}，"
      f"但每 token 只激活 top_k={moe.top_k} 个专家")
y.sum().backward()
print("MoE forward + backward OK")


## B.5 长上下文与 FlashAttention

注意力的真正瓶颈来自$O(N^2)$复杂度：序列变长时计算量和显存都会爆炸。常见优化有两类。

**算法层面：稀疏注意力。** 只让每个词元关注部分位置，把$O(N^2)$降下来：

- **Sliding Window Attention**：每个词元只关注前后$w$个邻居（Longformer、Mistral）；
- **Dilated Attention**：注意力窗口按层次扩张以兼顾局部和全局（LongNet）；
- **压缩注意力**：把远距离上下文压缩成少量摘要token（Compressive Transformer）。

**系统层面：FlashAttention（Dao等人，2022）。** 不改变注意力的数学语义，而是通过**分块计算+在线Softmax**避免在显存里存放完整的$N\times N$打分矩阵，让中间结果常驻GPU片上SRAM，端到端通常能提速2$\sim$4倍。PyTorch 2.0+已把它集成进`F.scaled_dot_product_attention`，受支持硬件会自动启用。


In [ ]:
import math

# 1) 内置 SDPA（在受支持硬件上会自动走 FlashAttention 内核）
torch.manual_seed(2021)
q = torch.randn(2, 4, 16, 8)
k = torch.randn(2, 4, 16, 8)
v = torch.randn(2, 4, 16, 8)
out = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # 因果掩码（生成式）
print("SDPA causal out:", tuple(out.shape))

# 与手写缩放点积 + 因果掩码对比，确认数学语义一致
d_k = q.size(-1)
scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
L = q.size(-2)
causal = torch.tril(torch.ones(L, L, dtype=torch.bool))        # 下三角=可见
scores = scores.masked_fill(~causal, torch.finfo(scores.dtype).min)
manual = F.softmax(scores, dim=-1) @ v
print("SDPA == 手写因果注意力:", torch.allclose(out, manual, atol=1e-5))


# 2) 稀疏注意力示意：滑动窗口掩码（每个 query 只看前后 w 个邻居）
def sliding_window_mask(L, w):
    """返回 [L, L] 布尔掩码，True=可见：|i-j| <= w。"""
    i = torch.arange(L)[:, None]
    j = torch.arange(L)[None, :]
    return (i - j).abs() <= w


mask = sliding_window_mask(16, w=2)
print("滑动窗口掩码 shape:", tuple(mask.shape),
      " 每行可见数(中部):", int(mask[8].sum()))   # 中间位置应为 2w+1=5
# 把窗口掩码喂给 SDPA：attn_mask 为布尔时 True=参与、False=屏蔽
out_win = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
print("滑动窗口 SDPA out:", tuple(out_win.shape))


## 小结

本节把章末的现代Transformer优化逐一落到可运行代码：

- **Pre-LN**形成清晰残差流，训练稳定，是现代大模型默认；Post-LN上限略高但难训。
- **RoPE**用旋转注入位置，相对位置天然成立、外推友好、零额外参数。
- **三种架构范式**里，decoder-only凭结构简单、目标统一、推理高效成为生成式主流。
- **MoE**用稀疏路由在保持容量的同时压低单步计算量。
- **FlashAttention/稀疏注意力**从系统和算法两条路缓解$O(N^2)$瓶颈；PyTorch的`scaled_dot_product_attention`是工程首选。

这些组件正是Llama、Qwen、GPT等模型的工程基石——下一章《大语言模型与智能体》会直接复用它们，从零搭起一个nanoGPT。
